# Sentiment Analysis for Complaints
## Using a Word List Based Approach

In this notebook I am analyzing the emotional tone of student complaints. Instead of training an ML model I am using a word list based approach where I have manually created lists of positive and negative words with intensity scores.

**How it works:**
- I have a list of negative words like angry frustrated terrible each with a score from -1 to -3
- I have a list of positive words like thank appreciate helpful each with a score from +1 to +3
- If a word like not or never appears before another word it flips the polarity
- Words like very or extremely amplify the intensity of the next word
- At the end I calculate the average score per word and classify as Positive Neutral or Negative

The sentiment also influences complaint priority. If a complaint sounds angry or frustrated the priority gets automatically bumped up.


---
## 1. Loading the Sentiment Analyzer

In [ ]:
import sys, os, csv, json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.join('..', 'train'))
from sentiment import analyze_sentiment, sentiment_priority_boost

print('Sentiment analyzer loaded (from scratch, no ML)')

---
## 2. Testing with Some Sample Complaints

In [ ]:
test_texts = [
    'Thank you for your help, really appreciate the quick response',
    'The problem is still not fixed, very disappointed with the service',
    'WiFi is not working in the library',
    'This is absolutely ridiculous and unacceptable behavior from the staff',
    'Please look into the water leakage issue in my room',
    'I hate this college, worst experience ever, useless administration',
    'Great work by the maintenance team, very helpful',
    'exam schedule not released yet',
    'someone stole my laptop from library',
]

print(f'{"Text":<55s} {"Label":<18s} {"Score":>7s} {"Neg":>4s} {"Pos":>4s}')
print('-' * 90)
for t in test_texts:
    r = analyze_sentiment(t)
    sub = r['sub_label']
    prio_after = sentiment_priority_boost(r['label'], 'Medium')
    print(f'{t[:52]:<55s} {sub:<18s} {r["score"]:>6.2f}  {r["neg_words"]:>3d}  {r["pos_words"]:>3d}')

---
## 3. Running Sentiment on the Full Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'TrainDataset', 'processed_dataset_4500.csv')

with open(DATA_PATH, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Total complaints: {len(rows)}')

sentiment_dist = Counter()
priority_boosts = 0
sample_results = []

for r in rows[:1000]:  # Sample first 1000 for speed
    text = r['text']
    result = analyze_sentiment(text)
    sentiment_dist[result['label']] += 1
    sample_results.append((text[:60], result))

print('\nSentiment distribution (sampled 1000 complaints):')
total = sum(sentiment_dist.values())
for label in ['Positive', 'Neutral', 'Negative']:
    count = sentiment_dist[label]
    print(f'  {label:<10s}: {count:>4d} ({count/total*100:>5.1f}%)')

### Visualize Sentiment Distribution

In [ ]:
labels = ['Positive', 'Neutral', 'Negative']
counts = [sentiment_dist[l] for l in labels]
colors = ['#22c55e', '#6b7280', '#ef4444']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(labels, counts, color=colors, width=0.5)
ax1.set_ylabel('Count')
ax1.set_title('Sentiment Distribution (1000 samples)')
for i, v in enumerate(counts):
    ax1.text(i, v + 5, str(v), ha='center', fontweight='bold')

wedges, texts, autotexts = ax2.pie(counts, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title('Sentiment Distribution')

plt.tight_layout()
plt.show()

---
## 4. Looking at Examples from Each Sentiment Type

In [ ]:
print(f'{"Complaint":<60s} {"Label":<18s} {"Sub-Label":<18s} {"Score":>7s}')
print('=' * 105)

# Show examples from each sentiment class
for target_label in ['Negative', 'Neutral', 'Positive']:
    examples = [(t, r) for t, r in sample_results if r['label'] == target_label]
    print(f'\n--- {target_label.upper()} ({len(examples)} samples) ---')
    for text, r in examples[:3]:
        print(f'{text[:57]:<60s} {r["label"]:<18s} {r["sub_label"]:<18s} {r["score"]:>6.2f}')

---
## 5. How Sentiment Affects Priority

In [ ]:
# Show how sentiment would affect priority boosting
print('How sentiment affects priority (starting from Medium):')
print(f'{"Sentiment":<12s} {"Starting":<10s} {"After Boost":<12s}')
print('-' * 36)
for s in ['Negative', 'Neutral', 'Positive']:
    boosted = sentiment_priority_boost(s, 'Medium')
    print(f'{s:<12s} {"Medium":<10s} {boosted:<12s}')
    
print()
print('Full priority boost matrix:')
print(f'{"Sentiment":<12s} {"Low ->":<10s} {"Medium ->":<10s} {"High ->":<10s}')
print('-' * 42)
for s in ['Negative', 'Neutral', 'Positive']:
    l = sentiment_priority_boost(s, 'Low')
    m = sentiment_priority_boost(s, 'Medium')
    h = sentiment_priority_boost(s, 'High')
    print(f'{s:<12s} {l:<10s} {m:<10s} {h:<10s}')

---
## Summary

### Sentiment Analysis Approach
- **Method:** Lexicon-based (no training data required)
- **Positive words:** ~50 words with intensity scores (+1 to +3)
- **Negative words:** ~100 words with intensity scores (-1 to -3)
- **Negation handling:** `not`, `never`, `no` flip next word's polarity
- **Intensifiers:** `very`, `extremely`, `absolutely` amplify scores

### Integration
- Auto-detect sentiment on complaint submission
- Priority auto-boost for negative/angry complaints
- Sentiment badge visible in admin and student dashboards
- Score stored in DB for reporting